# Bio-Hermes-001 EDA — NeuroFusion-AD Phase 2A
**Date**: 2026-03-11 | **N**: 945 participants with amyloid classification
Warning: No PHI displayed — participant IDs hashed before any logging

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for kernel execution
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
EDA_DIR = 'C:/Users/SachinM/desktop/neurofusion-ad/notebooks/eda'
print("Imports OK")

Imports OK


In [2]:
train = pd.read_csv(r'C:/Users/SachinM/desktop/neurofusion-ad/data/processed/biohermes/biohermes001_train.csv')
val   = pd.read_csv(r'C:/Users/SachinM/desktop/neurofusion-ad/data/processed/biohermes/biohermes001_val.csv')
all_bh = pd.concat([train, val], ignore_index=True)
print(f"Total: {len(all_bh)} participants")
print(f"Amyloid+: {all_bh['AMYLOID_POSITIVE'].mean():.1%}")
print(f"Age: {all_bh['AGE'].mean():.1f} +/- {all_bh['AGE'].std():.1f}")
print(f"Columns: {all_bh.columns.tolist()}")

Total: 945 participants
Amyloid+: 36.2%
Age: 72.0 +/- 6.7
Columns: ['USUBJID', 'AMYLOID_POSITIVE', 'AGE', 'SEX_CODE', 'RACE', 'EDUCATION_YEARS', 'PTAU217', 'ABETA40_PLASMA', 'ABETA42_PLASMA', 'GFAP_PLASMA', 'NFL_PLASMA', 'PTAU181_PLASMA', 'ABETA4240_RATIO', 'MMSE_BASELINE', 'acoustic_delayed_recall', 'acoustic_object_recall', 'acoustic_image_descr_score', 'acoustic_intraword_pause', 'acoustic_speaking_rate', 'acoustic_verbal_fluency', 'acoustic_image_speaking_rate', 'acoustic_naming_duration', 'acoustic_monotonicity', 'acoustic_pause_rate', 'motor_dcr_clock_score', 'motor_dcr_delayed_recall', 'motor_dcr_score', 'motor_digit_naming_median_rt', 'motor_digit_naming_rt', 'motor_digit_naming_throughput', 'motor_digit_naming_pct_correct', 'motor_sdmt_acc', 'motor_sdmt_attempted', 'motor_spiral_ccw_dom', 'motor_spiral_ccw_nondom', 'motor_spiral_cw_dom', 'motor_spiral_cw_nondom', 'motor_trails_b_acc', 'motor_trails_b_time', 'MMSE_SLOPE', 'TIME_TO_EVENT', 'EVENT_INDICATOR']


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
counts = all_bh['AMYLOID_POSITIVE'].value_counts()
axes[0].bar(['Negative','Positive'], [counts.get(0,0), counts.get(1,0)], color=['#2196F3','#F44336'])
axes[0].set_title(f'Amyloid Positivity (N={len(all_bh)})')
axes[0].set_ylabel('Count')
pos = int(all_bh['AMYLOID_POSITIVE'].sum())
axes[0].text(1, pos+5, f'{pos/len(all_bh):.1%}', ha='center')

sex_counts = all_bh.groupby(['SEX_CODE','AMYLOID_POSITIVE']).size().unstack(fill_value=0)
sex_counts.index = ['Female','Male']
sex_counts.columns = ['Amyloid-','Amyloid+']
sex_counts.plot(kind='bar', ax=axes[1], color=['#2196F3','#F44336'], rot=0)
axes[1].set_title('Sex by Amyloid Status')
plt.tight_layout()
plt.savefig(f'{EDA_DIR}/fig_biohermes_class_balance.png', dpi=100, bbox_inches='tight')
plt.show()

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for amy, label, color in [(0,'Amyloid-','#2196F3'),(1,'Amyloid+','#F44336')]:
    sub = all_bh[all_bh['AMYLOID_POSITIVE']==amy]['PTAU217'].dropna()
    axes[0].hist(sub.values.tolist(), alpha=0.7, label=f'{label} (n={len(sub)})', color=color, bins=25)
axes[0].set_xlabel('pTau-217 (scaled)')
axes[0].set_title('Plasma pTau-217 by Amyloid Status\n(Lilly immunoassay)')
axes[0].legend()

for amy, label, color in [(0,'Amyloid-','#2196F3'),(1,'Amyloid+','#F44336')]:
    sub = all_bh[all_bh['AMYLOID_POSITIVE']==amy]['ABETA4240_RATIO'].dropna()
    axes[1].hist(sub.values.tolist(), alpha=0.7, label=f'{label} (n={len(sub)})', color=color, bins=25)
axes[1].set_xlabel('Abeta42/40 ratio (scaled)')
axes[1].set_title('Plasma Abeta42/40 Ratio by Amyloid Status\n(Roche Diagnostics)')
axes[1].legend()
plt.tight_layout()
plt.savefig(f'{EDA_DIR}/fig_biohermes_ptau_abeta.png', dpi=100, bbox_inches='tight')
plt.show()

print("pTau-217 mean by amyloid status:")
print(all_bh.groupby('AMYLOID_POSITIVE')['PTAU217'].agg(['mean','std']).round(3))

pTau-217 mean by amyloid status:
                   mean    std
AMYLOID_POSITIVE              
0.0              -0.001  0.462
1.0               0.909  0.915


In [5]:
acoustic_cols = [c for c in all_bh.columns if c.startswith('acoustic_')]
print(f"Acoustic features available ({len(acoustic_cols)}): {acoustic_cols}")
if acoustic_cols:
    n_show = min(len(acoustic_cols), 10)
    n_rows = (n_show + 4) // 5
    fig, axes = plt.subplots(n_rows, 5, figsize=(18, 4 * n_rows))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for i, col in enumerate(acoustic_cols[:n_show]):
        ax = axes_flat[i]
        for amy, label, color in [(0,'Amyloid-','#2196F3'),(1,'Amyloid+','#F44336')]:
            sub = all_bh[all_bh['AMYLOID_POSITIVE']==amy][col].dropna()
            if len(sub) > 0:
                ax.hist(sub.values.tolist(), alpha=0.7, label=label, color=color, bins=20)
        ax.set_title(col.replace('acoustic_',''), fontsize=8)
        ax.tick_params(labelsize=7)
    for i in range(n_show, len(axes_flat)):
        axes_flat[i].set_visible(False)
    axes_flat[0].legend(fontsize=7)
    plt.suptitle('Acoustic Features by Amyloid Status (Bio-Hermes-001 - Real Data)')
    plt.tight_layout()
    plt.savefig(f'{EDA_DIR}/fig_biohermes_acoustic.png', dpi=100, bbox_inches='tight')
    plt.show()

Acoustic features available (10): ['acoustic_delayed_recall', 'acoustic_object_recall', 'acoustic_image_descr_score', 'acoustic_intraword_pause', 'acoustic_speaking_rate', 'acoustic_verbal_fluency', 'acoustic_image_speaking_rate', 'acoustic_naming_duration', 'acoustic_monotonicity', 'acoustic_pause_rate']


In [6]:
motor_cols = [c for c in all_bh.columns if c.startswith('motor_')]
print(f"Motor features available ({len(motor_cols)}): {motor_cols}")
if motor_cols:
    n_show = min(len(motor_cols), 15)
    n_rows = (n_show + 4) // 5
    fig, axes = plt.subplots(n_rows, 5, figsize=(18, 4 * n_rows))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for i, col in enumerate(motor_cols[:n_show]):
        ax = axes_flat[i]
        for amy, label, color in [(0,'Amyloid-','#2196F3'),(1,'Amyloid+','#F44336')]:
            sub = all_bh[all_bh['AMYLOID_POSITIVE']==amy][col].dropna()
            if len(sub) > 0:
                ax.hist(sub.values.tolist(), alpha=0.7, label=label, color=color, bins=20)
        ax.set_title(col.replace('motor_',''), fontsize=8)
        ax.tick_params(labelsize=7)
    for i in range(n_show, len(axes_flat)):
        axes_flat[i].set_visible(False)
    axes_flat[0].legend(fontsize=7)
    plt.suptitle('Motor/Cognitive Features by Amyloid Status (Bio-Hermes-001 - Real Data)')
    plt.tight_layout()
    plt.savefig(f'{EDA_DIR}/fig_biohermes_motor.png', dpi=100, bbox_inches='tight')
    plt.show()

Motor features available (15): ['motor_dcr_clock_score', 'motor_dcr_delayed_recall', 'motor_dcr_score', 'motor_digit_naming_median_rt', 'motor_digit_naming_rt', 'motor_digit_naming_throughput', 'motor_digit_naming_pct_correct', 'motor_sdmt_acc', 'motor_sdmt_attempted', 'motor_spiral_ccw_dom', 'motor_spiral_ccw_nondom', 'motor_spiral_cw_dom', 'motor_spiral_cw_nondom', 'motor_trails_b_acc', 'motor_trails_b_time']


In [7]:
if 'RACE' in all_bh.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    race_counts = all_bh.groupby(['RACE','AMYLOID_POSITIVE']).size().unstack(fill_value=0)
    race_counts.columns = ['Amyloid-','Amyloid+']
    race_counts.plot(kind='bar', ax=ax, color=['#2196F3','#F44336'], rot=45)
    ax.set_title('Race/Ethnicity by Amyloid Status (Fairness Analysis)')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.savefig(f'{EDA_DIR}/fig_biohermes_race.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(race_counts)
else:
    print("RACE column not found in data")

                                           Amyloid-  Amyloid+
RACE                                                         
AMERICAN INDIAN OR ALASKA NATIVE                  2         0
ASIAN                                            11         6
BLACK OR AFRICAN AMERICAN                        76        29
NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER         1         0
UNKNOWN                                           8         0
WHITE                                           505       307


In [8]:
# Bio-Hermes pTau correlation with acoustic features
acoustic_cols = [c for c in all_bh.columns if c.startswith('acoustic_')]
if acoustic_cols:
    corr_cols = ['PTAU217', 'ABETA4240_RATIO', 'MMSE_BASELINE', 'AMYLOID_POSITIVE'] + acoustic_cols[:6]
    corr_cols = [c for c in corr_cols if c in all_bh.columns]
    corr = all_bh[corr_cols].corr()
    fig, ax = plt.subplots(figsize=(12, 9))
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask)] = True
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                mask=mask, ax=ax, square=True, annot_kws={'size': 7})
    ax.set_title('Fluid + Acoustic Feature Correlations (Bio-Hermes-001)')
    plt.tight_layout()
    plt.savefig(f'{EDA_DIR}/fig_biohermes_correlation.png', dpi=100, bbox_inches='tight')
    plt.show()

In [9]:
# Final summary stats
n_total = len(all_bh)
amyloid_rate = all_bh['AMYLOID_POSITIVE'].mean()
age_mean = all_bh['AGE'].mean()
age_std = all_bh['AGE'].std()
acoustic_cols = [c for c in all_bh.columns if c.startswith('acoustic_')]
motor_cols = [c for c in all_bh.columns if c.startswith('motor_')]
ptau_pos = all_bh[all_bh['AMYLOID_POSITIVE']==1]['PTAU217'].mean()
ptau_neg = all_bh[all_bh['AMYLOID_POSITIVE']==0]['PTAU217'].mean()

print("=" * 55)
print("EDA SUMMARY - BIO-HERMES-001")
print("=" * 55)
print(f"Total participants:         {n_total}")
print(f"Amyloid+ rate:              {amyloid_rate:.1%}")
print(f"Age:                        {age_mean:.1f} +/- {age_std:.1f} years")
print(f"Acoustic features:          {len(acoustic_cols)} (REAL - Aural Analytics)")
print(f"Motor features:             {len(motor_cols)} (REAL - Linus Health)")
print(f"pTau-217 data:              REAL (Lilly immunoassay)")
print(f"pTau-217 mean (amy+):       {ptau_pos:.3f}")
print(f"pTau-217 mean (amy-):       {ptau_neg:.3f}")
print(f"Longitudinal labels:        NOT AVAILABLE (cross-sectional)")
print(f"MMSE slope:                 NaN (cross-sectional design)")
print(f"Time to event:              NaN (cross-sectional design)")
print()
print("Key advantage over ADNI: Bio-Hermes-001 provides REAL plasma")
print("pTau-217 and REAL digital biomarkers, making it the primary source")
print("for acoustic/motor encoder training.")

EDA SUMMARY - BIO-HERMES-001
Total participants:         945
Amyloid+ rate:              36.2%
Age:                        72.0 +/- 6.7 years
Acoustic features:          10 (REAL - Aural Analytics)
Motor features:             15 (REAL - Linus Health)
pTau-217 data:              REAL (Lilly immunoassay)
pTau-217 mean (amy+):       0.909
pTau-217 mean (amy-):       -0.001
Longitudinal labels:        NOT AVAILABLE (cross-sectional)
MMSE slope:                 NaN (cross-sectional design)
Time to event:              NaN (cross-sectional design)

Key advantage over ADNI: Bio-Hermes-001 provides REAL plasma
pTau-217 and REAL digital biomarkers, making it the primary source
for acoustic/motor encoder training.


## EDA Summary — Bio-Hermes-001

| Metric | Value |
|--------|-------|
| Total participants | 945 |
| Amyloid+ rate | ~36% |
| pTau-217 data | **REAL** (Lilly immunoassay) |
| Acoustic data | **REAL** (Aural Analytics) |
| Motor data | **REAL** (Linus Health) |
| Longitudinal labels | Not available (cross-sectional) |

**Key advantage over ADNI**: Bio-Hermes-001 provides real plasma pTau-217 and real digital biomarkers, making it the primary source for acoustic/motor encoder training.

**Fairness note**: Race/ethnicity analysis included for regulatory bias assessment. Any disparities flagged here must be addressed in the Risk Management File (RMF-001).